# 01 - 检查 Chen / PopAlign PBMC 药物扰动数据

来源: https://doi.org/10.6084/m9.figshare.11837097

RISE预处理后参考规模: 29,433细胞 × 9,461基因 × 46条件（仅用于核对）

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('..')
data_dir = base / 'data' / 'chen_popalign_pbmc'
print(f'数据目录: {data_dir.resolve()}')

# 查看文件清单
manifest = data_dir / 'metadata' / 'figshare_file_list.json'
if manifest.exists():
    with open(manifest) as f:
        m = json.load(f)
    print(f'\nfigshare文件数: {len(m.get("files", []))}')
    for f in m.get('files', []):
        size = f.get('size', 0)
        print(f'  {f.get("name")} ({size/1024/1024:.1f} MB)')
else:
    print('文件清单不存在，请先运行 download_chen_popalign.py --metadata-only')

## 检查表达矩阵

In [ ]:
# 查找已下载的表达矩阵
h5ad_files = list((data_dir / 'processed').glob('*.h5ad'))
rds_files = list((data_dir / 'processed').glob('*.rds'))
csv_files = list((data_dir / 'processed').glob('*.csv*'))

print(f'h5ad: {len(h5ad_files)}')
print(f'rds: {len(rds_files)}')
print(f'csv: {len(csv_files)}')

if h5ad_files:
    import anndata
    adata = anndata.read_h5ad(h5ad_files[0])
    print(f'\n形状: {adata.shape} (细胞 × 基因)')
    print(f'X dtype: {adata.X.dtype}')
    print(f'obs列: {list(adata.obs.columns)}')
    print(f'var列: {list(adata.var.columns)}')
    print(f'表达类型: {adata.uns.get("expression_type", "未标记")}')
elif csv_files:
    df = pd.read_csv(csv_files[0], index_col=0)
    print(f'\n形状: {df.shape}')
    print(f'前5行索引: {list(df.index[:5])}')
    print(f'前5列: {list(df.columns[:5])}')
else:
    print('\n未找到表达矩阵文件。请先下载数据。')
    print('参考规模(仅核对): 29433细胞 × 9461基因 × 46条件')

## 检查药物条件

In [ ]:
# 如果有元数据，检查药物条件
meta_files = list((data_dir / 'metadata').glob('*sample*'))
if meta_files:
    print(f'元数据文件: {[f.name for f in meta_files]}')
else:
    print('样本元数据尚未下载。')
    print('预期包含: 药物名称、剂量、细胞类型、对照条件')
    print('RISE参考: 46个条件')